In [1]:
from transitivity_sims.models import EquallySpacedBTL, Tournament
from transitivity_sims.metrics import (
    TrialMetrics, kendall_tau_correlation, spearman_correlation,
    pairwise_accuracy, adjacent_pair_accuracy, top_k_accuracy
)
from transitivity_sims.experiment import (
    ExperimentConfig, ExperimentResult, ExperimentRunner,
    run_experiment_grid
)
from transitivity_sims.plots import generate_all_plots
 
import numpy as np


In [2]:
from transitivity_sims.experiment import _USE_VIBERANK_RC
print(f"Using viberank RC class: {_USE_VIBERANK_RC}")
if not _USE_VIBERANK_RC:
    print("  (Falling back to standalone RC implementation — same algorithm)")
 
model = EquallySpacedBTL(n=10, beta=0.5)
rng = np.random.default_rng(42)
tourn = model.sample_tournament(rng)
 
print(f"\nModel: {model}")
print(f"Adjacent flip prob: {model.adjacent_flip_probability:.3f}")
print(f"Win matrix shape: {tourn.win_matrix.shape}")
print(f"Circular triads: {tourn.circular_triad_count}")
print(f"Normalized cycle rate (zeta): {tourn.normalized_cycle_rate:.3f}")
 
from transitivity_sims.experiment import rc_scores_from_tournament, scores_to_ranking
 
scores = rc_scores_from_tournament(tourn)
ranking = scores_to_ranking(scores)
true_ranking = model.true_ranking
 
print(f"\nRC scores:  {np.round(scores, 4)}")
print(f"RC ranking: {ranking}")
print(f"True ranking: {true_ranking}")
print(f"Kendall tau: {kendall_tau_correlation(ranking, true_ranking):.3f}")
print(f"Spearman rho: {spearman_correlation(ranking, true_ranking):.3f}")


Using viberank RC class: False
  (Falling back to standalone RC implementation — same algorithm)

Model: EquallySpacedBTL(n=10, beta=0.5000, adj_flip=0.378)
Adjacent flip prob: 0.378
Win matrix shape: (10, 10)
Circular triads: 7
Normalized cycle rate (zeta): 0.175

RC scores:  [0.2381 0.0476 0.1905 0.381  0.0476 0.0952 0.     0.     0.     0.    ]
RC ranking: [ 2  5  3  1  6  4  7  8  9 10]
True ranking: [ 1  2  3  4  5  6  7  8  9 10]
Kendall tau: 0.733
Spearman rho: 0.855


In [3]:
print(f"\nWin counts per item (out-degrees): {tourn.out_degrees.astype(int)}")
print(f"Any item won all matches: {np.any(tourn.out_degrees == tourn.n - 1)}")


Win counts per item (out-degrees): [7 6 7 8 5 6 3 1 1 1]
Any item won all matches: False


In [4]:
print(f"RC scores: {scores}")
print(f"Min score: {scores.min():.10f}")
print(f"Max score: {scores.max():.10f}")
print(f"All positive: {(scores > 0).all()}")

RC scores: [2.38095237e-01 4.76190480e-02 1.90476196e-01 3.80952371e-01
 4.76190480e-02 9.52380995e-02 8.07793567e-29 2.86985925e-43
 2.86985925e-43 2.86985925e-43]
Min score: 0.0000000000
Max score: 0.3809523715
All positive: True


In [5]:
n_values = [20, 50, 100]
beta_values = [0.01, 0.05, 0.10, 0.15, 0.20, 0.30,
               0.50, 0.75, 1.0, 1.5, 2.0, 3.0]
 
# Print what each beta means
print(f"{'beta':>6s}  {'adj_flip':>8s}  {'b (n=20)':>12s}  {'b (n=50)':>12s}")
print("-" * 45)
for beta in beta_values:
    flip = 1 / (1 + np.exp(beta))
    b20 = np.exp(beta * 19)
    b50 = np.exp(beta * 49)
    print(f"{beta:6.2f}  {flip:8.3f}  {b20:12.1f}  {b50:12.1f}")
 




  beta  adj_flip      b (n=20)      b (n=50)
---------------------------------------------
  0.01     0.498           1.2           1.6
  0.05     0.488           2.6          11.6
  0.10     0.475           6.7         134.3
  0.15     0.463          17.3        1556.2
  0.20     0.450          44.7       18033.7
  0.30     0.426         298.9     2421747.6
  0.50     0.378       13359.7  43673179097.6
  0.75     0.321     1544174.5  9126877256863956.0
  1.00     0.269   178482301.0  1907346572495099789312.0
  1.50     0.182  2384474784797.7  83299888461860515254013402284032.0
  2.00     0.119  31855931757113756.0  3637970947608804749879905722463248516644864.0
  3.00     0.047  5685719999335932014624768.0  6938871417758403719029073689814301559277489968133780679665123328.0


In [6]:
results = run_experiment_grid(
    n_values=n_values,
    beta_values=beta_values,
    n_trials=500,
    n_consistency_trials=200,
    seed=42,
    verbose=True,
)



  n = 20
  EquallySpacedBTL(n=20, beta=0.0100, adj_flip=0.498) ... done (0.4s) | zeta=0.864, tau=0.084, rho_s=0.120, adj_acc=0.505
  EquallySpacedBTL(n=20, beta=0.0500, adj_flip=0.488) ... done (0.4s) | zeta=0.813, tau=0.380, rho_s=0.526, adj_acc=0.528
  EquallySpacedBTL(n=20, beta=0.1000, adj_flip=0.475) ... done (0.5s) | zeta=0.683, tau=0.585, rho_s=0.764, adj_acc=0.556
  EquallySpacedBTL(n=20, beta=0.1500, adj_flip=0.463) ... done (0.5s) | zeta=0.542, tau=0.669, rho_s=0.842, adj_acc=0.569
  EquallySpacedBTL(n=20, beta=0.2000, adj_flip=0.450) ... done (0.5s) | zeta=0.419, tau=0.716, rho_s=0.877, adj_acc=0.586
  EquallySpacedBTL(n=20, beta=0.3000, adj_flip=0.426) ... done (0.5s) | zeta=0.249, tau=0.761, rho_s=0.908, adj_acc=0.600
  EquallySpacedBTL(n=20, beta=0.5000, adj_flip=0.378) ... done (0.6s) | zeta=0.099, tau=0.831, rho_s=0.946, adj_acc=0.645
  EquallySpacedBTL(n=20, beta=0.7500, adj_flip=0.321) ... done (0.7s) | zeta=0.040, tau=0.898, rho_s=0.975, adj_acc=0.711
  EquallySpace

In [7]:
print(f"\n{'n':>3s} {'beta':>6s} {'flip':>6s} {'zeta':>6s} "
      f"{'tau':>8s} {'rho_s':>8s} {'adj':>6s} {'pair':>6s} {'cons_t':>6s}")
print("-" * 65)
for r in results:
    s = r.summary()
    print(f"{s['n']:3.0f} {s['beta']:6.3f} {s['adj_flip_prob']:6.3f} "
          f"{s['mean_cycle_rate']:6.3f} "
          f"{s['mean_kendall_tau']:6.3f}±{s['std_kendall_tau']:.3f} "
          f"{s['mean_spearman_rho']:6.3f}   "
          f"{s['mean_adjacent_acc']:6.3f} {s['mean_pairwise_acc']:6.3f} "
          f"{s['mean_consistency_tau']:6.3f}")



  n   beta   flip   zeta      tau    rho_s    adj   pair cons_t
-----------------------------------------------------------------
 20  0.010  0.498  0.864  0.084±0.156  0.120    0.505  0.542  0.021
 20  0.050  0.488  0.813  0.380±0.123  0.526    0.528  0.690  0.187
 20  0.100  0.475  0.683  0.585±0.086  0.764    0.556  0.792  0.431
 20  0.150  0.463  0.542  0.669±0.070  0.842    0.569  0.834  0.540
 20  0.200  0.450  0.419  0.716±0.064  0.877    0.586  0.858  0.593
 20  0.300  0.426  0.249  0.761±0.055  0.908    0.600  0.881  0.659
 20  0.500  0.378  0.099  0.831±0.046  0.946    0.645  0.915  0.742
 20  0.750  0.321  0.040  0.898±0.036  0.975    0.711  0.949  0.836
 20  1.000  0.269  0.019  0.932±0.028  0.986    0.767  0.966  0.890
 20  1.500  0.182  0.006  0.964±0.020  0.994    0.848  0.982  0.939
 20  2.000  0.119  0.002  0.978±0.015  0.997    0.900  0.989  0.961
 20  3.000  0.047  0.000  0.991±0.010  0.999    0.955  0.995  0.985
 50  0.010  0.498  0.927  0.311±0.085  0.448    0.509

In [8]:
 
generate_all_plots(results, output_dir='./figures')
 



Generating plots...
  Saved: ./figures/beta_vs_accuracy.png
  Saved: ./figures/cycle_rate_vs_accuracy.png
  Saved: ./figures/scatter_trials.png
  Saved: ./figures/self_consistency.png
  Saved: ./figures/top_k_recovery.png
  All plots generated.


In [9]:
# Pick a mid-range beta to look at trial-level distributions
beta_inspect = 0.3
n_inspect = 50
 
r = [r for r in results if r.beta == beta_inspect and r.n == n_inspect][0]
s = r.summary()
 
print(f"Config: n={n_inspect}, beta={beta_inspect}")
print(f"  Adjacent flip prob: {s['adj_flip_prob']:.3f}")
print(f"  Mean cycle rate (zeta): {s['mean_cycle_rate']:.3f}")
print(f"  Mean Kendall tau: {s['mean_kendall_tau']:.3f} ± {s['std_kendall_tau']:.3f}")
print(f"  Mean Spearman rho: {s['mean_spearman_rho']:.3f}")
print(f"  Mean adjacent acc: {s['mean_adjacent_acc']:.3f}")
print(f"  Mean top-k acc: {s['mean_top_k_acc']:.3f}")
print(f"  Mean consistency tau: {s['mean_consistency_tau']:.3f}")
print(f"  Mean consistency rho: {s['mean_consistency_rho']:.3f}")


Config: n=50, beta=0.3
  Adjacent flip prob: 0.426
  Mean cycle rate (zeta): 0.060
  Mean Kendall tau: 0.877 ± 0.018
  Mean Spearman rho: 0.975
  Mean adjacent acc: 0.594
  Mean top-k acc: 0.822
  Mean consistency tau: 0.818
  Mean consistency rho: 0.951


In [10]:

 
print(f"\n{'n':>3s} {'beta':>6s} {'E[T_n]':>10s} {'mean(T_n)':>10s} {'std(T_n)':>10s}")
print("-" * 45)
for r in results:
    if not np.isnan(r.expected_cycles):
        obs_mean = np.mean(r._gather('cycle_count'))
        obs_std = np.std(r._gather('cycle_count'))
        print(f"{r.n:3d} {r.beta:6.3f} {r.expected_cycles:10.1f} "
              f"{obs_mean:10.1f} {obs_std:10.1f}")



  n   beta     E[T_n]  mean(T_n)   std(T_n)
---------------------------------------------
 20  0.010      284.3      285.0       14.5
 20  0.050      267.5      268.4       19.1
 20  0.100      225.8      225.3       24.4
 20  0.150      178.9      178.9       25.2
 20  0.200      138.1      138.3       23.6
 20  0.300       82.2       82.1       18.2
 20  0.500       33.5       32.6       11.2
 20  0.750       13.9       13.3        5.9
 20  1.000        6.7        6.3        3.9
 20  1.500        2.0        1.8        1.7
 20  2.000        0.7        0.6        0.9
 20  3.000        0.1        0.1        0.3
 50  0.010     4823.2     4818.7       74.6
 50  0.050     3515.0     3511.8      151.2
 50  0.100     1907.6     1908.3      133.1
 50  0.150     1075.6     1079.7       96.0
 50  0.200      661.9      664.5       73.9
 50  0.300      307.7      312.3       43.3
 50  0.500      104.7      104.8       21.2
 50  0.750       40.2       39.6       10.8
 50  1.000       18.9       1